# Stage 2: Exploratory Data Analysis (EDA)

## Objective
Explore the Bank Marketing dataset to understand its structure, distributions, relationships, class balance, and potential outliers before preprocessing and model development.

**Dataset:** `bank.csv`  
**Target:** `y`  
**Problem type:** Binary classification  
**Rows and columns:** 4,521 × 17

### Sections
1. Target variable distribution
2. Numerical feature analysis
3. Categorical feature analysis
4. Correlation analysis
5. Feature versus target analysis
6. Outlier analysis
7. EDA summary

> The original dataset is kept unchanged. The `duration` feature is explored here but will be excluded from predictive model training because it is only known after the customer interaction.

## 2.0 Imports and dataset loading

Load the dataset using a semicolon delimiter. The UCI Bank Marketing CSV uses `;` rather than a comma.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/raw/bank.csv", sep=";")

print("Dataset shape:", df.shape)
display(df.head())

## 2.1 Target variable distribution

### Objective
Measure the number and percentage of customers who subscribed (`yes`) and did not subscribe (`no`). This identifies class imbalance, which affects model evaluation and training decisions.

In [ ]:
target_counts = df["y"].value_counts()
target_percentages = df["y"].value_counts(normalize=True).mul(100)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages.round(2)
})
display(target_summary)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="y", hue="y", palette="viridis", legend=False)
ax.set_title("Target Variable Distribution")
ax.set_xlabel("Subscription Status")
ax.set_ylabel("Number of Customers")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

### Observation and ML implication

Expected from the initial validation:
- `no`: 4,000 records (approximately 88.48%).
- `yes`: 521 records (approximately 11.52%).

The target is imbalanced. Accuracy alone can be misleading, so model evaluation will also include precision, recall, F1, AUC, and MCC. Any resampling, if later justified, must be applied only to the training data.

## 2.2 Numerical feature analysis

### Objective
Review the descriptive statistics and distributions of numerical variables. This helps identify scale differences, skewness, zero-heavy variables, and extreme observations.

In [ ]:
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()
print("Numerical features:", numerical_columns)
display(df[numerical_columns].describe().T)

In [ ]:
df[numerical_columns].hist(
    figsize=(16, 12),
    bins=30,
    edgecolor="black"
)
plt.suptitle("Numerical Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()

### Observation prompts

Record which features appear skewed, have many zero values, or contain extreme values. Do not remove observations based on plots alone.

**ML implication:** Scaling will be important for distance- and margin-based models such as KNN, SVM, and Logistic Regression. Tree-based models generally do not require scaling.

## 2.3 Categorical feature analysis

### Objective
Inspect the frequency of each category, including `unknown` values. In this dataset, `unknown` is a category label, not a pandas null value.

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()
categorical_columns.remove("y")

print("Categorical features:", categorical_columns)

for col in categorical_columns:
    print(f"\nDistribution of {col}")
    display(df[col].value_counts(dropna=False).rename("Count").to_frame())

In [ ]:
for col in categorical_columns:
    plt.figure(figsize=(10, 4))
    order = df[col].value_counts().index

    sns.countplot(
        data=df,
        x=col,
        order=order,
        hue=col,
        palette="viridis",
        legend=False
    )

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### Observation prompts

Identify the most frequent categories, rare categories, and features with substantial `unknown` counts.

**ML implication:** Categorical variables require encoding. The initial preprocessing plan will retain `unknown` as a category and use one-hot encoding.

## 2.4 Correlation analysis

### Objective
Calculate Pearson correlations among numerical features and visualize them in a heatmap. Correlation measures linear association; it does not establish causation.

In [ ]:
correlation_matrix = df[numerical_columns].corr(method="pearson")
display(correlation_matrix.round(2))

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.show()

### Observation prompts

Look for pairs with relatively strong positive or negative correlations. Do not remove a feature solely because its correlation is low: nonlinear relationships and interactions may still be predictive.

## 2.5 Feature versus target analysis

### Objective
Compare feature distributions across the two target classes. These plots are exploratory and do not prove that a feature causes subscription.

### 2.5.1 Numerical features versus target

The plots below include `duration` for descriptive exploration. It will be excluded from model training to avoid target leakage because call duration is only known after the call.

In [ ]:
for col in numerical_columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="y", y=col, hue="y", palette="viridis", legend=False)
    plt.title(f"{col} by Subscription Status")
    plt.xlabel("Subscription Status")
    plt.tight_layout()
    plt.show()

### 2.5.2 Categorical features versus target

Count plots show raw counts. Since category sizes differ, also calculate the within-category subscription percentages.

In [ ]:
for col in categorical_columns:
    plt.figure(figsize=(11, 5))
    sns.countplot(data=df, x=col, hue="y", palette="viridis")
    plt.title(f"{col} by Subscription Status")
    plt.xlabel(col)
    plt.ylabel("Customer Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in categorical_columns:
    percentage_table = pd.crosstab(
        df[col],
        df["y"],
        normalize="index"
    ).mul(100)

    print(f"\nSubscription percentage by {col} (%)")
    display(percentage_table.round(2))

### Observation prompts

Record numerical and categorical features where the class distributions appear different. Prefer within-category percentages over raw counts when comparing categories of unequal sizes.

**ML implication:** These patterns can motivate model experiments, but final feature usefulness must be judged on held-out test data.

## 2.6 Outlier analysis using the IQR method

### Objective
Flag potential outliers in numerical features using the interquartile range (IQR).

\[
IQR = Q3 - Q1
\]

\[
Lower\ Bound = Q1 - 1.5(IQR)
\]

\[
Upper\ Bound = Q3 + 1.5(IQR)
\]

An observation outside these bounds is flagged as a potential outlier, not automatically an error.

In [ ]:
outlier_rows = []

for col in numerical_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Potential Outlier Count": int(count)
    })

outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary.round(2))

In [ ]:
for col in numerical_columns:
    plt.figure(figsize=(9, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Potential Outliers: {col}")
    plt.tight_layout()
    plt.show()

### Observation and decision

Review the flagged values in context. Some extreme values may be genuine customer characteristics. Do not automatically delete or cap them. Any outlier treatment should be justified and fitted using training data only.

## 2.7 EDA summary

Complete the table below after reviewing the outputs. Replace the prompts with observations from your actual run.

In [ ]:
eda_summary = pd.DataFrame([
    ["Target distribution", "No: 4,000 (88.48%); Yes: 521 (11.52%)", "Imbalanced target; use multiple evaluation metrics."],
    ["Missing values", "No pandas null values found during validation.", "No null-value imputation currently indicated."],
    ["Numerical features", "Record observations from histograms and describe().", "Assess scaling and skewness during preprocessing."],
    ["Categorical features", "Record frequent/rare categories and unknown counts.", "Encode categorical features; retain unknown initially."],
    ["Correlation", "Record notable relationships from heatmap.", "Investigate redundancy; correlation alone is not a removal rule."],
    ["Feature-target patterns", "Record differences observed in plots/tables.", "Validate predictive value using held-out data."],
    ["Potential outliers", "Record IQR flags and whether values seem plausible.", "Do not remove automatically; justify any treatment."]
], columns=["Analysis", "Observation", "ML Implication"])

display(eda_summary)

## 2.8 Important modelling note: target leakage

The `duration` feature records the duration of the last contact. It is only known after the call has taken place. If the intended prediction is made before contacting the customer, using `duration` would leak information unavailable at prediction time.

Therefore, `duration` is excluded from the model input features in Stage 3 and later. The original raw CSV remains unchanged.